# ¿Qué artistas va a escuchar por primera vez cada usuario?
Usamos el historial de Last.fm para armar una lista de artistas por usuario. Después miramos las escuchas del período siguiente para contar cuántos de esos artistas aparecieron de verdad.

**Cómo correrla.** Abrí esta notebook desde la carpeta `mineria/` y ejecutá las celdas en orden. La descarga de Last.fm queda en `05_descarga_lastfm.ipynb`. Esta notebook lee los archivos guardados, así que podés repetir las pruebas con otras semillas sin volver a consultar la API.

**Cómo leerla.** Primero armamos los datos. Después vemos cómo calcula sus recomendaciones cada modelo. Elegimos los parámetros en validación y recién al final evaluamos los modelos elegidos.

## 1. Preparar los datos
Tomamos escuchas desde enero de 2010. En validación usamos el historial anterior al 1 de octubre de 2013 y buscamos aciertos entre octubre y diciembre. En la prueba final ampliamos el historial hasta diciembre y buscamos aciertos entre enero y marzo de 2014. En ambos casos el día del corte ya pertenece al período que queremos predecir.

In [32]:
# %pip install duckdb pandas pyarrow numpy scipy scikit-learn implicit threadpoolctl

from config import CLEAN_DIR, conectar, ruta_sql
import re
import time
import unicodedata
from itertools import product

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Usamos 42 en validación y repetimos la prueba final con las tres semillas.
SEMILLAS = [42, 123, 2026]
CORTES_K = [3, 5, 10]
# Elegimos los parámetros según el ranking de cinco artistas.
K_REFERENCIA = 5
assert SEMILLAS and len(set(SEMILLAS)) == len(SEMILLAS)
assert CORTES_K and min(CORTES_K) >= 1 and len(set(CORTES_K)) == len(CORTES_K)
# Tamaño del catálogo por corte y máximo de usuarios evaluados por semilla.
N_ARTISTAS = 200
N_EVAL = 1000
MIN_ARTISTAS_TRAIN = 10
ITERACIONES = 30
FECHA_VAL = '2013-10-01'
FECHA_TEST = '2014-01-01'
FIN_TEST = '2014-04-01'

MAX_CANCIONES_ARTISTA = 2
ruta_parquet = CLEAN_DIR.parent / 'parquet'
archivo_artistas = ruta_parquet / 'artists.parquet'
archivo_tracks = ruta_parquet / 'tracks.parquet'

# Usamos la misma conexión y las mismas rutas que en la descarga.
con = conectar()
con.execute("SET threads = 4")
con.execute(f"""
    CREATE OR REPLACE VIEW events_clean AS
    SELECT * FROM {ruta_sql('events_clean')}
    WHERE timestamp >= epoch(TIMESTAMP '2010-01-01')
""")
con.execute(f"CREATE OR REPLACE VIEW acoustic_clean AS SELECT * FROM {ruta_sql('acoustic_clean')}")

def q(sql):
    """Ejecuta una consulta SQL y devuelve una tabla de pandas."""
    return con.sql(sql).df()
def unix(fecha):
    """Convierte una fecha UTC a segundos para compararla con los eventos."""
    # Los eventos guardan segundos desde 1970 en UTC.
    return int(pd.Timestamp(fecha, tz='UTC').timestamp())

def nombre_normalizado(valor):
    """Pasa un nombre a minúsculas y unifica acentos y espacios para poder cruzarlo."""
    valor = unicodedata.normalize('NFKD', str(valor).casefold())
    valor = ''.join(x for x in valor if not unicodedata.combining(x))
    return re.sub(r'\s+', ' ', valor).strip()

nombres_artistas = (pd.read_parquet(archivo_artistas, columns=['artist_id', 'artist_name'])
                    .dropna(subset=['artist_id']).drop_duplicates('artist_id')
                    .set_index('artist_id')['artist_name'])
print('Datos preparados. Artistas con nombre:', len(nombres_artistas))

Datos preparados. Artistas con nombre: 3190371


### Una matriz de usuarios por artistas
Una fila representa un usuario y una columna, un artista. Elegimos los 1.000 artistas con más oyentes distintos antes de cada corte. Guardamos tres versiones de las mismas escuchas.

- **Binaria** vale 1 si el usuario escuchó al artista.
- **Logarítmica** usa `log(1 + escuchas)`. Escuchar diez veces pesa más que escuchar una vez, pero la diferencia se achica.
- **Reciente** parte del valor logarítmico y le da más peso si una proporción alta de esas escuchas ocurrió en los últimos 90 días.

Evaluamos usuarios que ya habían escuchado al menos diez artistas del catálogo y que después escucharon al menos uno nuevo de ese catálogo. El conjunto de esos artistas nuevos es la respuesta que usamos para evaluar. Un artista «nuevo» es nuevo en el historial observado de ese usuario.

### Ejemplo de la clase 5 con escuchas
En clase usamos una tabla dinámica de usuarios y películas. Con música, cada casillero contiene cuántas veces el usuario escuchó al artista. Este ejemplo inventado muestra cómo se lee la matriz. Con todos los usuarios guardamos solo las posiciones con escuchas para ahorrar memoria.

In [33]:
ejemplo = pd.DataFrame({
    'usuario': ['Ana', 'Ana', 'Luis'],
    'artista': ['Radiohead', 'Muse', 'Muse'],
    'escuchas': [3, 1, 2]
})
display(pd.pivot_table(ejemplo, index='usuario', columns='artista',
                       values='escuchas', aggfunc='sum', fill_value=0))

artista,Muse,Radiohead
usuario,,
Ana,1,3
Luis,2,0


In [34]:
def preparar_datos(corte, fin):
    """Arma las matrices anteriores al corte y los artistas nuevos observados entre corte y fin."""
    inicio, final = unix(corte), unix(fin)
    hace_90 = inicio - 90 * 86400

    # Se usa solamente el pasado para elegir artistas y armar el train.
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE pares_train AS
        WITH catalogo AS (
            SELECT artist_id
            FROM events_clean
            WHERE timestamp < {inicio} AND artist_id IS NOT NULL AND user_id IS NOT NULL
            GROUP BY artist_id
            ORDER BY COUNT(DISTINCT user_id) DESC, artist_id
            LIMIT {N_ARTISTAS}
        )
        SELECT e.user_id, e.artist_id,
               COUNT(*) AS escuchas,
               COUNT(*) FILTER (WHERE e.timestamp >= {hace_90}) AS escuchas_90d
        FROM events_clean e
        JOIN catalogo c USING (artist_id)
        WHERE e.timestamp < {inicio} AND e.user_id IS NOT NULL
        GROUP BY e.user_id, e.artist_id
    """)
    train = q('SELECT * FROM pares_train')
    if train.empty:
        raise ValueError('El train quedó vacío; revisá las rutas y fechas')
    # Convertimos los ID originales en posiciones consecutivas de la matriz.
    filas, usuarios = pd.factorize(train.user_id, sort=True)
    columnas, artistas = pd.factorize(train.artist_id, sort=True)
    n, m = len(usuarios), len(artistas)

    # Todas las matrices usan EXACTAMENTE las mismas filas y columnas.
    escuchas = train.escuchas.to_numpy(dtype='float32')
    recientes = train.escuchas_90d.to_numpy(dtype='float32')
    ratio = recientes / np.maximum(escuchas, 1)
    def matriz(datos):
        """Ubica cada valor en la fila de su usuario y la columna de su artista."""
        return csr_matrix((np.asarray(datos, dtype='float32'), (filas, columnas)),
                          shape=(n, m))
    matrices = {
        'binaria': matriz(np.ones(len(train))),
        'log': matriz(np.log1p(escuchas)),
        'reciente': matriz(np.log1p(escuchas) * (0.7 + 0.8 * ratio)),
    }

    # Variables históricas por artista, sin utilizar la evaluación.
    resumen = (train.groupby('artist_id', as_index=False)
               .agg(oyentes=('user_id', 'nunique'),
                    escuchas=('escuchas', 'sum'),
                    escuchas_90d=('escuchas_90d', 'sum')))
    resumen['escuchas_por_oyente'] = resumen.escuchas / resumen.oyentes.clip(lower=1)
    resumen['proporcion_reciente'] = resumen.escuchas_90d / resumen.escuchas.clip(lower=1)

    # Objetivo: artistas nuevos del catálogo escuchados en el período posterior.
    objetivos = q(f"""
        WITH suficientes AS (
            SELECT user_id FROM pares_train GROUP BY user_id
            HAVING COUNT(*) >= {MIN_ARTISTAS_TRAIN}
        )
        SELECT DISTINCT e.user_id, e.artist_id
        FROM events_clean e
        JOIN suficientes s USING(user_id)
        WHERE e.timestamp >= {inicio} AND e.timestamp < {final}
          AND e.artist_id IN (SELECT DISTINCT artist_id FROM pares_train)
          AND NOT EXISTS (
              SELECT 1 FROM pares_train p
              WHERE p.user_id = e.user_id AND p.artist_id = e.artist_id
          )
    """)
    con.execute('DROP TABLE pares_train')
    if objetivos.empty:
        raise ValueError('No hay usuarios evaluables en este período')
    objetivos['fila'] = usuarios.get_indexer(objetivos.user_id)
    objetivos['columna'] = artistas.get_indexer(objetivos.artist_id)
    objetivos = objetivos[(objetivos.fila >= 0) & (objetivos.columna >= 0)]
    esperados = objetivos.groupby('fila').columna.apply(set).to_dict()
    densidad = 100 * matrices['binaria'].nnz / (n * m)
    print(f'{corte}: matriz {(n, m)}, densidad {densidad:.2f}%, usuarios elegibles {len(esperados)}')
    return dict(matrices=matrices, usuarios=usuarios, artistas=artistas,
                objetivos=esperados, resumen=resumen)

def muestra(d, semilla):
    """Selecciona hasta N_EVAL usuarios elegibles y devuelve sus artistas esperados."""
    objetivos = d['objetivos']
    # Ordenamos antes de sortear para que la semilla reproduzca la muestra.
    ids = np.array(sorted(objetivos))
    elegidos = np.random.default_rng(semilla).choice(ids, min(N_EVAL, len(ids)), replace=False)
    return {u: objetivos[u] for u in elegidos}

val = preparar_datos(FECHA_VAL, FECHA_TEST)
val['esperados'] = muestra(val, SEMILLAS[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2013-10-01: matriz (52629, 200), densidad 15.53%, usuarios elegibles 16544


## 2. Describir a los artistas
Leemos las etiquetas y los metadatos guardados por la notebook 05. Para usar etiquetas de canciones tomamos hasta dos canciones por artista, elegidas por sus escuchas anteriores al corte. Si falta `tracks.parquet`, ese bloque queda vacío. El audio se calcula por separado con los identificadores de las canciones y el archivo de características acústicas.

In [35]:
ARCHIVO_TAGS = ruta_parquet / 'artist_tags_lastfm_enriquecido.parquet'
CACHE_INFO = ruta_parquet / 'artist_info_actual_lastfm.parquet'
CACHE_SIM = ruta_parquet / 'artist_similares_actual_lastfm.parquet'
CACHE_TRACKS = ruta_parquet / 'track_info_y_tags_actual_lastfm.parquet'
def numero(x):
    """Convierte un valor a decimal y devuelve cero si falta o es inválido."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0

def leer_cache(archivo, columnas):
    """Lee las columnas disponibles de una descarga previa o devuelve una tabla vacía con el esquema pedido."""
    if archivo.exists():
        df = pd.read_parquet(archivo)
        if set(columnas).issubset(df):
            return df
    return pd.DataFrame(columns=columnas)

df_tags = leer_cache(ARCHIVO_TAGS, ['artist_id','artist_name','tag','peso'])
df_info = leer_cache(CACHE_INFO, ['artist_id','listeners_actual','playcount_actual'])
df_similares = leer_cache(CACHE_SIM, ['artist_id','similar_name','similitud'])
df_tracks = leer_cache(CACHE_TRACKS, ['artist_id','track_id','duracion_ms','tags_json'])

In [36]:
def canciones_historicas(artistas, corte):
    """Elige hasta dos canciones por artista según sus escuchas previas al corte, con el límite configurado."""
    if not archivo_tracks.exists():
        print('No encontré tracks.parquet; omito las consultas de canciones.')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    columnas = set(q(f"DESCRIBE SELECT * FROM '{archivo_tracks.as_posix()}'").column_name)
    if not {'track_id','track_name'}.issubset(columnas):
        print('tracks.parquet no tiene track_id y track_name; omito canciones.')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    con.register('catalogo_canciones', pd.DataFrame({'artist_id': np.asarray(artistas, dtype='int64')}))
    canciones = q(f"""
        WITH conteo AS (
            SELECT e.artist_id, e.track_id, COUNT(*) AS escuchas
            FROM events_clean e JOIN catalogo_canciones c USING(artist_id)
            WHERE e.timestamp < {unix(corte)} AND e.track_id IS NOT NULL
            GROUP BY e.artist_id, e.track_id
        ), orden AS (
            SELECT *, ROW_NUMBER() OVER(
                PARTITION BY artist_id ORDER BY escuchas DESC, track_id
            ) AS puesto
            FROM conteo
        )
        SELECT o.artist_id, o.track_id, t.track_name, o.escuchas
        FROM orden o JOIN '{archivo_tracks.as_posix()}' t USING(track_id)
        WHERE o.puesto <= {MAX_CANCIONES_ARTISTA}
    """)
    con.unregister('catalogo_canciones')
    canciones['artist_name'] = canciones.artist_id.map(nombres_artistas)
    return canciones.dropna(subset=['artist_name', 'track_name'])

### Pasar de etiquetas a géneros
Agrupamos etiquetas como `hard rock` y `classic rock` en una misma columna. Sumamos sus pesos y normalizamos por artista. Para las etiquetas de canciones, también pesa cuántas veces se escuchó esa canción antes del corte. Una fila en cero significa que no obtuvimos etiquetas reconocidas.

In [37]:
# Podés ampliar este diccionario, pero evitá cambiarlo mirando el resultado de test.
MAPA_GENEROS = {
    'rock': ['rock', 'classic rock', 'hard rock', 'soft rock', 'progressive rock', 'psychedelic rock',
             'garage rock', 'rock and roll', 'glam rock', 'southern rock', 'art rock', 'post-rock'],
    'pop': ['pop', 'dance pop', 'pop rock', 'synthpop', 'synth pop', 'electropop', 'power pop', 'k-pop', 'j-pop'],
    'metal': ['metal', 'heavy metal', 'thrash metal', 'death metal', 'black metal', 'power metal',
              'doom metal', 'progressive metal', 'nu metal', 'speed metal', 'metalcore', 'sludge metal'],
    'indie_alternative': ['indie', 'indie rock', 'indie pop', 'alternative', 'alternative rock',
                          'alternative pop', 'britpop', 'shoegaze', 'dream pop', 'grunge',
                          'post-grunge', 'lo-fi', 'lofi', 'post punk revival'],
    'punk': ['punk', 'punk rock', 'post-punk', 'pop punk', 'hardcore punk', 'hardcore', 'emo', 'screamo'],
    'hip_hop': ['hip-hop', 'hip hop', 'rap', 'trap', 'gangsta rap', 'alternative hip hop',
                'conscious hip hop', 'grime', 'drill'],
    'electronic': ['electronic', 'electronica', 'house', 'deep house', 'techno', 'trance',
                   'dubstep', 'drum and bass', 'edm', 'electro', 'trip-hop', 'trip hop', 'idm'],
    'rnb_soul': ['rnb', 'r&b', 'rhythm and blues', 'soul', 'neo-soul', 'neo soul',
                 'funk', 'disco', 'motown'],
    'jazz': ['jazz', 'smooth jazz', 'bebop', 'swing', 'fusion', 'jazz fusion'],
    'blues': ['blues', 'blues rock', 'delta blues'],
    'folk_country': ['folk', 'indie folk', 'folk rock', 'country', 'bluegrass', 'americana', 'singer-songwriter'],
    'latin': ['latin', 'latin pop', 'reggaeton', 'salsa', 'bachata', 'tango', 'cumbia',
              'bossa nova', 'samba', 'latin rock'],
    'reggae': ['reggae', 'ska', 'dub', 'dancehall'],
    'classical': ['classical', 'classical music', 'baroque', 'opera', 'orchestral',
                  'contemporary classical', 'romantic'],
    'ambient': ['ambient', 'new age', 'downtempo', 'chillout', 'chill', 'experimental',
                'noise', 'drone', 'soundtrack'],
}
GENEROS = list(MAPA_GENEROS)
TAG_A_GENERO = {nombre_normalizado(t): genero
                for genero, etiquetas in MAPA_GENEROS.items() for t in etiquetas}

In [38]:
def matriz_generos(artist_ids, tags):
    """Suma el peso de las etiquetas de cada artista por género y normaliza cada fila."""
    tabla = tags[['artist_id', 'tag', 'peso']].copy()
    tabla['genero'] = tabla.tag.map(lambda x: TAG_A_GENERO.get(nombre_normalizado(x)))
    tabla['peso'] = pd.to_numeric(tabla.peso, errors='coerce').fillna(0).clip(lower=0)
    tabla = tabla.dropna(subset=['genero'])
    if tabla.empty:
        return np.zeros((len(artist_ids), len(GENEROS)), dtype='float32')
    tabla = (tabla.groupby(['artist_id', 'genero']).peso.sum().unstack(fill_value=0)
             .reindex(index=list(artist_ids), columns=GENEROS, fill_value=0))
    X = tabla.to_numpy(dtype='float32')
    X = X / np.maximum(X.sum(axis=1, keepdims=True), 1)
    return X

def matriz_generos_tracks(artist_ids, metadatos_tracks, candidatos):
    """Resume los géneros de las canciones de cada artista, ponderados por sus escuchas."""
    import json
    idx = {int(a): i for i,a in enumerate(artist_ids)}
    X = np.zeros((len(artist_ids), len(GENEROS)), dtype='float32')
    escuchas = candidatos.set_index('track_id').escuchas.to_dict() if len(candidatos) else {}
    for t in metadatos_tracks.itertuples(index=False):
        i = idx.get(int(t.artist_id))
        if i is None:
            continue
        try:
            etiquetas = json.loads(t.tags_json) if isinstance(t.tags_json, str) else []
        except (ValueError, TypeError):
            etiquetas = []
        for etiqueta in etiquetas:
            g = TAG_A_GENERO.get(nombre_normalizado(etiqueta.get('tag','')))
            if g is not None:
                X[i, GENEROS.index(g)] += numero(etiqueta.get('peso')) * np.log1p(escuchas.get(t.track_id, 0))
    return X / np.maximum(X.sum(axis=1, keepdims=True), 1)

X_generos_val = matriz_generos(val['artistas'], df_tags)
print('Géneros definidos:', len(GENEROS), GENEROS)
print('Artistas con géneros reconocidos:', (X_generos_val.sum(axis=1)>0).sum(), '/', len(val['artistas']))
print('Etiquetas originales sin mapeo (se omiten):',
      df_tags.loc[~df_tags.tag.map(lambda t: nombre_normalizado(t) in TAG_A_GENERO), 'tag'].nunique())

Géneros definidos: 15 ['rock', 'pop', 'metal', 'indie_alternative', 'punk', 'hip_hop', 'electronic', 'rnb_soul', 'jazz', 'blues', 'folk_country', 'latin', 'reggae', 'classical', 'ambient']
Artistas con géneros reconocidos: 199 / 200
Etiquetas originales sin mapeo (se omiten): 975


### Audio, historial y datos actuales
Para el audio, cada canción distinta del artista cuenta una vez. Calculamos la media y el desvío de ocho características. En el historial resumimos oyentes, cantidad de escuchas y actividad reciente. Usamos el logaritmo para moderar valores muy altos y estandarizamos las columnas numéricas para combinar escalas distintas.

La popularidad y los artistas similares consultados en 2026 quedan en variantes exploratorias. También las etiquetas de género provienen de la consulta a Last.fm: podemos describir los artistas con ellas, pero desconocemos cuáles eran exactamente sus etiquetas en 2013.

In [39]:
AUDIO = ['energy', 'valence', 'danceability', 'acousticness',
         'instrumentalness', 'tempo', 'speechiness', 'loudness']

def escalar(tabla):
    """Completa faltantes y lleva las columnas numéricas a una escala comparable."""
    """Escala solo columnas disponibles, rellenando ausentes con la mediana."""
    if tabla.shape[1] == 0:
        return np.zeros((len(tabla), 0), dtype='float32')
    X = tabla.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median()).fillna(0).to_numpy(dtype='float32')
    return StandardScaler().fit_transform(X).astype('float32')

def audio_por_artista(artistas, corte):
    """Resume las características de sus canciones distintas conocidas antes del corte."""
    con.register('catalogo_audio', pd.DataFrame({'artist_id': np.asarray(artistas, dtype='int64')}))
    columnas = ', '.join([f'AVG(a.{f}) AS {f}_media, STDDEV_POP(a.{f}) AS {f}_std' for f in AUDIO])
    df = q(f"""
        WITH canciones AS (
            SELECT DISTINCT artist_id, track_id
            FROM events_clean JOIN catalogo_audio USING(artist_id)
            WHERE timestamp < {unix(corte)} AND track_id IS NOT NULL
        )
        SELECT artist_id, COUNT(*) AS canciones_con_audio, {columnas}
        FROM canciones JOIN acoustic_clean a USING(track_id)
        GROUP BY artist_id
    """)
    con.unregister('catalogo_audio')
    return df.set_index('artist_id').reindex(artistas)

def historial_por_artista(d):
    """Recupera la actividad previa de cada artista en el orden del catálogo."""
    df = d['resumen'].set_index('artist_id').reindex(d['artistas'])
    df = df[['oyentes', 'escuchas', 'escuchas_por_oyente', 'proporcion_reciente']].copy()
    for col in ['oyentes', 'escuchas', 'escuchas_por_oyente']:
        df[col] = np.log1p(df[col])
    return df

def similitud_artistas(artistas, similares):
    """Arma una fila por artista con las similitudes que informó Last.fm."""
    # Cada artista se describe por los artistas del catálogo a los que Last.fm lo relaciona.
    m = len(artistas)
    X = np.zeros((m, m), dtype='float32')
    indices = {int(a): i for i,a in enumerate(artistas)}
    nombres = {nombre_normalizado(nombres_artistas.get(int(a), '')): i
               for i,a in enumerate(artistas)}
    for s in similares.itertuples(index=False):
        i = indices.get(int(s.artist_id))
        j = nombres.get(nombre_normalizado(s.similar_name))
        if i is not None and j is not None and i != j:
            X[i, j] = max(X[i, j], numero(s.similitud))
    return X

def columnas_actuales(artistas, info):
    """Prepara los contadores actuales de Last.fm para la comparación exploratoria."""
    df = info.drop_duplicates('artist_id').set_index('artist_id').reindex(artistas)
    columnas = ['listeners_actual', 'playcount_actual']
    df = df.reindex(columns=columnas).apply(pd.to_numeric, errors='coerce')
    return np.log1p(df.clip(lower=0))

In [40]:
def construir_features(d, corte, canciones_base):
    """Arma los bloques de características en el orden de las columnas de la matriz de escuchas."""
    artistas = d['artistas']

    audio = audio_por_artista(artistas, corte)
    historico = historial_por_artista(d)
    generos = matriz_generos(artistas, df_tags)
    tracks = df_tracks.merge(canciones_base[['artist_id', 'track_id']], on=['artist_id', 'track_id'], how='inner')
    generos_tracks = matriz_generos_tracks(artistas, tracks, canciones_base)

    # Duración: metadato de canciones, independiente de su popularidad actual.
    # La media se calcula sobre las canciones históricas consultadas.
    duracion = (tracks[tracks['artist_id'].isin(artistas)]
                .groupby('artist_id').duracion_ms.agg(['mean', 'std'])
                .rename(columns={'mean': 'duracion_media_ms', 'std': 'duracion_std_ms'})
                .reindex(artistas)) if len(tracks) else pd.DataFrame(
                    0, index=artistas, columns=['duracion_media_ms', 'duracion_std_ms'])

    # Las columnas actuales se mantienen separadas de las históricas.
    grupos = {
        'audio': escalar(audio.drop(columns=['canciones_con_audio'], errors='ignore')),
        'generos': generos,
        'generos_canciones': generos_tracks,
        'historico': escalar(pd.concat([historico,
                  np.log1p(audio[['canciones_con_audio']].fillna(0)),
                  duracion], axis=1)),
        'similares_actuales': similitud_artistas(artistas, df_similares),
        'popularidad_actual': escalar(columnas_actuales(artistas, df_info)),
    }
    return grupos, dict(audio=audio, historico=historico, info=df_info,
                        generos=generos, generos_canciones=generos_tracks, duracion=duracion)

# Las canciones se eligieron solamente con el historial previo al 01/10/2013.
candidatas_tracks = canciones_historicas(val['artistas'], FECHA_VAL)
grupos_val, detalle_val = construir_features(val, FECHA_VAL, candidatas_tracks)
for nombre, X in grupos_val.items():
    assert X.shape[0] == val['matrices']['log'].shape[1], nombre
    print(f'{nombre:21s} -> {X.shape[1]} columnas')

No encontré tracks.parquet; omito las consultas de canciones.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

audio                 -> 16 columnas
generos               -> 15 columnas
generos_canciones     -> 15 columnas
historico             -> 7 columnas
similares_actuales    -> 200 columnas
popularidad_actual    -> 2 columnas


## 3. Armar y evaluar una lista de recomendaciones
Todos los modelos pasan por la misma regla. Primero ordenamos los artistas por puntaje y excluimos los que el usuario ya escuchó. Si faltan candidatos con puntaje válido, completamos con artistas populares que todavía no aparecen en la lista.

In [41]:
def top_k(puntajes, escuchados, popularidad, k):
    """Devuelve hasta k artistas nuevos y usa popularidad si faltan puntajes."""
    p = np.array(puntajes, dtype='float64', copy=True)
    p[escuchados] = -np.inf
    orden = np.argsort(-p, kind='stable')
    lista = orden[np.isfinite(p[orden])][:k]
    if len(lista) < k:
        respaldo = np.argsort(-popularidad, kind='stable')
        respaldo = respaldo[~np.isin(respaldo, np.r_[escuchados, lista])]
        lista = np.r_[lista, respaldo[:k - len(lista)]]
    return lista

### Qué cuenta como acierto
Imaginá que recomendamos cinco artistas y el usuario después escuchó dos de ellos. Si en ese período descubrió ocho artistas del catálogo, su precisión es 2/5 y su recall es 2/8. Su hit rate individual vale 1 porque hubo al menos un acierto. El NDCG además mira el orden: acertar en el primer puesto vale más que acertar en el quinto.

Promediamos esas cuatro medidas por usuario. La cobertura se calcula para el conjunto: ¿qué porcentaje del catálogo apareció al menos una vez entre todas las recomendaciones?

In [42]:
def metricas_usuario(lista, esperados, k):
    """Compara una lista con los artistas que el usuario escuchó por primera vez en el período."""
    aciertos = np.isin(lista, list(esperados))
    precision = aciertos.sum() / k
    recall = aciertos.sum() / len(esperados)
    # Descontamos puntos a los aciertos que quedan más abajo en la lista.
    dcg = np.sum(aciertos / np.log2(np.arange(2, len(lista) + 2)))
    # Dividimos por el mejor orden posible para este usuario.
    ideal = np.sum(1 / np.log2(np.arange(2, min(k, len(esperados)) + 2)))
    return precision, recall, dcg / ideal, int(aciertos.any())


def medir_listas(rankings, esperados, n_artistas, k):
    """Promedia las métricas por usuario y calcula qué parte del catálogo fue recomendada."""
    # Para K=3 miramos los tres primeros del mismo ranking usado para K=5 y K=10.
    por_usuario = []
    artistas_recomendados = set()
    for usuario, ranking in rankings.items():
        lista = ranking[:k]
        por_usuario.append(metricas_usuario(lista, esperados[usuario], k))
        artistas_recomendados.update(lista)

    precision, recall, ndcg, hit_rate = np.mean(por_usuario, axis=0)
    return dict(precision=precision, recall=recall, ndcg=ndcg, hit_rate=hit_rate,
                cobertura_pct=100 * len(artistas_recomendados) / n_artistas)

### Modelo 1 · Popularidad
Contamos cuántos usuarios escucharon a cada artista antes del corte. Todos parten del mismo orden de popularidad, y cada usuario recibe los primeros artistas que todavía no escuchó. Es la referencia más simple.

In [43]:
def popularidad(d):
    """Cuenta los oyentes distintos de cada artista en entrenamiento."""
    return np.asarray(d['matrices']['binaria'].sum(axis=0)).ravel()

### Modelo 2 · ALS
Como ampliación del práctico de la clase 5, probamos ALS. Aprende un vector de números para cada usuario y otro para cada artista a partir de sus escuchas. Alterna dos pasos: ajusta los vectores de los usuarios con los de artistas fijos y después hace lo inverso. Para ordenar artistas, multiplica el vector del usuario por el de cada artista y suma los productos.

**Factores** indica cuántos números tiene cada vector. **Alpha** regula cuánto confiamos en una escucha observada. **Regularización** penaliza vectores con valores muy grandes. En esta implementación calculamos la confianza antes de entrenar y dejamos el multiplicador interno de la biblioteca en 1.

In [44]:
def entrenar_als(matriz, factores, alpha, reg, semilla):
    """Ajusta ALS sobre la matriz de confianza y devuelve el modelo entrenado."""
    # Cuanto más alto el valor de la matriz, mayor confianza en ese artista.
    confianza = matriz.copy()
    confianza.data = 1 + alpha * confianza.data
    modelo = AlternatingLeastSquares(
        factors=factores, regularization=reg, iterations=ITERACIONES,
        random_state=semilla, alpha=1, num_threads=4, use_gpu=False)
    with threadpool_limits(limits=1, user_api='blas'):
        modelo.fit(confianza)
    return modelo

def puntajes_als(modelo, u):
    """Calcula un puntaje por artista a partir del vector del usuario."""
    return modelo.item_factors @ modelo.user_factors[u]

### Modelo 3 · Contenido
Representamos a cada artista con sus características. Después armamos un perfil del usuario promediando los artistas que ya escuchó, con más peso para los que escuchó más veces. La similitud coseno indica qué artistas se parecen a ese perfil. Probamos distintos grupos de características para ver qué aporta cada combinación.

In [45]:
def combinar_grupos(grupos, pesos):
    """Une las características elegidas aplicando el peso de cada bloque."""
    # sqrt(peso) equilibra la importancia de grupos previamente normalizados.
    bloques = []
    for nombre, peso in pesos.items():
        X = grupos[nombre]
        if X.shape[1] and peso > 0:
            bloques.append(np.sqrt(peso) * normalize(X, axis=1))
    if not bloques:
        raise ValueError('No quedaron columnas para combinar')
    return normalize(np.hstack(bloques).astype('float32'), axis=1)

def puntajes_contenido(matriz, X, u):
    """Compara el perfil musical del usuario con cada artista mediante similitud coseno."""
    fila = matriz[u]
    if not len(fila.indices):
        return np.full(X.shape[0], -np.inf)
    # Los índices indican qué artistas escuchó. Los valores indican su peso en el perfil.
    perfil = np.average(X[fila.indices], axis=0, weights=fila.data)
    if np.linalg.norm(perfil) == 0:
        return np.full(X.shape[0], -np.inf)
    return cosine_similarity(X, perfil.reshape(1, -1)).ravel()


def recomendar_contenido(d, X, u, pop, k):
    """Ordena los puntajes de contenido y devuelve los primeros k artistas nuevos."""
    matriz = d['matrices']['log']
    return top_k(puntajes_contenido(matriz, X, u), matriz[u].indices, pop, k)

### Modelo 4 · Híbrido
Como segunda ampliación, ordenamos los artistas una vez con ALS y otra por contenido. Convertimos cada orden a una escala entre 0 y 1 para poder combinarlos. Con peso 0,15, el contenido aporta el 15 % del puntaje combinado y ALS el 85 %. Cuando el perfil de contenido está vacío usamos los puntajes de ALS.

In [46]:
def rangos(p, conocidos):
    """Expresa la posición de cada candidato en una escala de cero a uno."""
    validos = np.isfinite(p)
    validos[conocidos] = False
    r = np.zeros(len(p), dtype='float32')
    if validos.sum():
        r[validos] = (rankdata(p[validos], method='average') - 1) / max(validos.sum()-1, 1)
    return r


def recomendar_hibrido(d, modelo, X, u, pop, peso, k):
    """Combina los rankings de ALS y contenido y devuelve los primeros k artistas nuevos."""
    matriz = d['matrices']['log']
    conocidos = matriz[u].indices
    p_als = puntajes_als(modelo, u)
    p_cont = puntajes_contenido(matriz, X, u)
    if not np.isfinite(p_cont).any():
        return top_k(p_als, conocidos, pop, k)
    combinado = (1-peso) * rangos(p_als, conocidos) + peso * rangos(p_cont, conocidos)
    return top_k(combinado, conocidos, pop, k)

### Modelo 5 · Colaborativo por vecinos
Buscamos usuarios con historiales parecidos mediante la similitud coseno de sus escuchas logarítmicas. Para puntuar un artista miramos cuánto lo escuchó cada vecino respecto de su propio promedio. Los vecinos más parecidos tienen más peso. Exigimos que al menos tres vecinos hayan escuchado al artista. En validación comparamos usar hasta 10, 25, 50, 75 o 100 vecinos.

In [47]:
def preparar_colaborativo(matriz):
    """Calcula las medias por usuario y las matrices necesarias para comparar vecinos."""
    cantidad = matriz.getnnz(axis=1)
    medias = np.asarray(matriz.sum(axis=1)).ravel() / np.maximum(cantidad, 1)
    # Restamos el promedio solo a los artistas que ese vecino escuchó.
    centrada = matriz.copy()
    centrada.data -= np.repeat(medias, cantidad)
    binaria = matriz.copy()
    binaria.data[:] = 1
    return medias, centrada, binaria, normalize(matriz)

def calcular_vecinos(normalizada, esperados, max_vecinos):
    """Busca los usuarios más parecidos a cada usuario evaluado y guarda sus similitudes."""
    cache = {}
    for u in sorted(esperados):
        # Como las filas ya tienen longitud 1, el producto calcula la similitud coseno.
        sim = (normalizada @ normalizada[u].T).toarray().ravel()
        # Un usuario no puede ser su propio vecino.
        sim[u] = 0
        orden = np.argsort(-sim, kind='stable')[:max_vecinos]
        orden = orden[sim[orden] > 0]
        cache[u] = (orden, sim[orden])
    return cache

def recomendar_colaborativo(d, pop, colab, cache, u, vecinos_max, k):
    """Puntúa artistas con los historiales de los vecinos y devuelve una lista ordenada."""
    medias, centrada, binaria, _ = colab
    vecinos, pesos = cache[u]
    vecinos, pesos = vecinos[:vecinos_max], pesos[:vecinos_max]
    p = np.full(len(pop), -np.inf)
    if len(vecinos):
        # Cada vecino aporta su diferencia con su propia media, ponderada por similitud.
        numerador = np.asarray(centrada[vecinos].T @ pesos).ravel()
        # Para cada artista sumamos solo los pesos de quienes lo escucharon.
        denominador = np.asarray(binaria[vecinos].T @ pesos).ravel()
        suficientes = binaria[vecinos].getnnz(axis=0) >= 3
        validos = (denominador > 0) & suficientes
        p[validos] = medias[u] + numerador[validos] / denominador[validos]
    return top_k(p, d['matrices']['log'][u].indices, pop, k)

## 4. Elegir configuraciones en validación
Usamos los mismos usuarios (semilla 42) y cinco recomendaciones para comparar todas las variantes. Elegimos por NDCG@5 porque interesa que los aciertos aparezcan cerca del principio de la lista. Cada función de recomendación recibe un usuario y devuelve sus artistas en orden.

In [48]:
CONFIG_CONTENIDO = {
    'audio': {'audio': 1},
    'generos': {'generos': 1},
    'audio+generos': {'audio': 0.60, 'generos': 0.40},
    'audio+generos+historia': {'audio': 0.45, 'generos': 0.35, 'historico': 0.20},
    'audio+generos+tracks': {'audio': 0.40, 'generos': 0.35, 'generos_canciones': 0.25},
    'todo_historico+tags': {'audio': 0.30, 'generos': 0.30,
                            'generos_canciones': 0.20, 'historico': 0.20},
    # Las dos últimas usan señales externas ACTUALES y se interpretan aparte.
    'exploratorio+similares_2026': {'audio': 0.25, 'generos': 0.25,
                                    'historico': 0.15, 'generos_canciones': 0.15,
                                    'similares_actuales': 0.20},
    'exploratorio+popularidad_2026': {'audio': 0.25, 'generos': 0.25,
                                      'historico': 0.15, 'generos_canciones': 0.15,
                                      'popularidad_actual': 0.20},
}


def evaluar_validacion(nombre, recomendar):
    """Aplica una función de recomendación a los usuarios de validación y resume sus métricas."""
    esperados = val['esperados']
    rankings = {u: recomendar(u) for u in sorted(esperados)}
    medidas = medir_listas(rankings, esperados, len(val['artistas']), K_REFERENCIA)
    return {'modelo': nombre, 'k': K_REFERENCIA, **medidas}

In [49]:
pop_val = popularidad(val)
val_resultados = []
modelos_contenido_val = {}
val_resultados.append(evaluar_validacion('Popularidad',
    lambda u: top_k(pop_val, val['matrices']['log'][u].indices, pop_val, K_REFERENCIA)))

# Calculamos los 100 vecinos una vez y probamos cinco tamaños de vecindario.
colab_val = preparar_colaborativo(val['matrices']['log'])
vecinos_val = calcular_vecinos(colab_val[3], val['esperados'], 100)
resultados_vecinos = []
for cant_vecinos in [10, 25, 50, 75, 100]:
    r = evaluar_validacion(f'Colaborativo ({cant_vecinos} vecinos)',
        lambda u, k=cant_vecinos: recomendar_colaborativo(
            val, pop_val, colab_val, vecinos_val, u, k, K_REFERENCIA))
    resultados_vecinos.append(dict(r, vecinos=cant_vecinos))
val_resultados.extend(resultados_vecinos)
tabla_vecinos = pd.DataFrame(resultados_vecinos)
display(tabla_vecinos[['vecinos', 'ndcg', 'hit_rate']].round(4))
mejor_fila = tabla_vecinos['ndcg'].idxmax()
mejor_cant_vecinos = int(tabla_vecinos.loc[mejor_fila, 'vecinos'])


for nombre, pesos in CONFIG_CONTENIDO.items():
    X = combinar_grupos(grupos_val, pesos)
    modelos_contenido_val[nombre] = X
    r = evaluar_validacion('Contenido: '+nombre,
                  lambda u, p=X: recomendar_contenido(val, p, u, pop_val, K_REFERENCIA))
    val_resultados.append(r)

,vecinos,ndcg,hit_rate
0,10,0.1169,0.313
1,25,0.1072,0.295
2,50,0.1064,0.298
3,75,0.1118,0.295
4,100,0.1178,0.306


In [50]:
# Búsqueda de parámetros de ALS (36 combinaciones). Si querés menos tiempo,
# sacá opciones de las listas, sin cambiar el período de validación.
# product arma todas las combinaciones en el orden indicado.
PARAMETROS_ALS = list(product(['binaria', 'log', 'reciente'], [16, 32, 64], [1, 10], [0.01, 0.1]))
resultados_als = []
mejor_als = None
mejor_ndcg = -1.0
for j, (senal, factores, alpha, reg) in enumerate(PARAMETROS_ALS, 1):
    inicio = time.perf_counter()
    modelo = entrenar_als(val['matrices'][senal], factores, alpha, reg, SEMILLAS[0])
    r = evaluar_validacion(f'ALS {senal} f{factores} a{alpha} r{reg}',
       lambda u: top_k(puntajes_als(modelo, u), val['matrices']['log'][u].indices, pop_val, K_REFERENCIA))
    resultados_als.append(dict(r, senal=senal, factores=factores, alpha=alpha, reg=reg))
    if r['ndcg'] > mejor_ndcg:
        mejor_ndcg = r['ndcg']
        mejor_als = modelo
        mejor_als_config = dict(senal=senal, factores=factores, alpha=alpha, reg=reg)
    print(f'ALS {j}/{len(PARAMETROS_ALS)}: {r["ndcg"]:.4f} - {time.perf_counter()-inicio:.1f}s')

val_resultados.extend(resultados_als)
tabla_als_val = pd.DataFrame(resultados_als).sort_values('ndcg', ascending=False)
print('Mejor configuración ALS:', mejor_als_config)
tabla_als_val.head(10).round(4)

  0%|          | 0/30 [00:00<?, ?it/s]

ALS 1/36: 0.1453 - 10.1s


  0%|          | 0/30 [00:00<?, ?it/s]

ALS 2/36: 0.1432 - 10.3s


  0%|          | 0/30 [00:00<?, ?it/s]

ALS 3/36: 0.1330 - 10.5s


... (63 outputs left out)

ALS 35/36: 0.0942 - 15.3s


  0%|          | 0/30 [00:00<?, ?it/s]

ALS 36/36: 0.0965 - 15.2s
Mejor configuración ALS: {'senal': 'reciente', 'factores': 16, 'alpha': 1, 'reg': 0.1}


,modelo,k,precision,recall,ndcg,hit_rate,cobertura_pct,senal,factores,alpha,reg
25,ALS reciente f16 a1 r0.1,5,0.1000,0.1446,0.1459,0.374,100.0,reciente,16,1,0.10
0,ALS binaria f16 a1 r0.01,5,0.0960,0.1499,0.1453,0.364,100.0,binaria,16,1,0.01
24,ALS reciente f16 a1 r0.01,5,0.0988,0.1421,0.1449,0.365,100.0,reciente,16,1,0.01
13,ALS log f16 a1 r0.1,5,0.0994,0.1434,0.1445,0.372,100.0,log,16,1,0.10
12,ALS log f16 a1 r0.01,5,0.0978,0.1449,0.1438,0.364,100.0,log,16,1,0.01
1,ALS binaria f16 a1 r0.1,5,0.0960,0.1462,0.1432,0.360,100.0,binaria,16,1,0.10
28,ALS reciente f32 a1 r0.01,5,0.0954,0.1408,0.1365,0.365,100.0,reciente,32,1,0.01
29,ALS reciente f32 a1 r0.1,5,0.0948,0.1403,0.1364,0.364,100.0,reciente,32,1,0.10
16,ALS log f32 a1 r0.01,5,0.0952,0.1435,0.1363,0.370,100.0,log,32,1,0.01
17,ALS log f32 a1 r0.1,5,0.0948,0.1378,0.1351,0.363,100.0,log,32,1,0.10


### Elegir el híbrido y el modelo de contenido
Probamos tres pesos para cada variante de contenido. Guardamos todas las métricas. Para elegir qué llevar a la prueba final excluimos las variantes exploratorias que usan popularidad o similitudes medidas en 2026.

In [51]:
# En el híbrido probamos todas las variantes de contenido y tres pesos.
# La configuración se decide mirando solo validación.
resultados_hibridos = []
mejor_hibrido = None
mejor_hibrido_ndcg = -1.0
for nombre, X in modelos_contenido_val.items():
    for peso in [0.15, 0.30, 0.50]:
        r = evaluar_validacion(f'Híbrido: {nombre}, peso={peso}',
           lambda u, p=X, w=peso: recomendar_hibrido(val, mejor_als, p, u, pop_val, w, K_REFERENCIA))
        resultados_hibridos.append(dict(r, contenido=nombre, peso=peso))
        # Los experimentos con información actual se reportan, pero no se eligen
        # como ganador del pronóstico retrospectivo de 2014.
        if 'exploratorio' not in nombre and r['ndcg'] > mejor_hibrido_ndcg:
            mejor_hibrido_ndcg = r['ndcg']
            mejor_hibrido = dict(contenido=nombre, peso=peso)

val_resultados.extend(resultados_hibridos)
tabla_val = pd.DataFrame(val_resultados).sort_values('ndcg', ascending=False)
print('Mejor híbrido:', mejor_hibrido)
print('Mejor contenido (sin popularidad/similares de 2026):')
solo_contenido = pd.DataFrame(val_resultados)
solo_contenido = solo_contenido[
    solo_contenido.modelo.str.startswith('Contenido:') &
    ~solo_contenido.modelo.str.contains('exploratorio')
]
mejor_contenido = solo_contenido.sort_values('ndcg', ascending=False).iloc[0]['modelo'].split(': ',1)[1]
print(mejor_contenido)
tabla_val.head(20).round(4)

Mejor híbrido: {'contenido': 'audio+generos+historia', 'peso': 0.15}
Mejor contenido (sin popularidad/similares de 2026):
todo_historico+tags


,modelo,k,precision,recall,ndcg,hit_rate,cobertura_pct,vecinos,senal,factores,alpha,reg,contenido,peso
39,ALS reciente f16 a1 r0.1,5,0.1000,0.1446,0.1459,0.374,100.0,NaN,reciente,16.0,1.0,0.10,NaN,NaN
14,ALS binaria f16 a1 r0.01,5,0.0960,0.1499,0.1453,0.364,100.0,NaN,binaria,16.0,1.0,0.01,NaN,NaN
38,ALS reciente f16 a1 r0.01,5,0.0988,0.1421,0.1449,0.365,100.0,NaN,reciente,16.0,1.0,0.01,NaN,NaN
27,ALS log f16 a1 r0.1,5,0.0994,0.1434,0.1445,0.372,100.0,NaN,log,16.0,1.0,0.10,NaN,NaN
26,ALS log f16 a1 r0.01,5,0.0978,0.1449,0.1438,0.364,100.0,NaN,log,16.0,1.0,0.01,NaN,NaN
15,ALS binaria f16 a1 r0.1,5,0.0960,0.1462,0.1432,0.360,100.0,NaN,binaria,16.0,1.0,0.10,NaN,NaN
71,"Híbrido: exploratorio+popularidad_2026, peso=0.15",5,0.1008,0.1476,0.1414,0.371,99.5,NaN,NaN,NaN,NaN,NaN,exploratorio+popularidad_2026,0.15
68,"Híbrido: exploratorio+similares_2026, peso=0.15",5,0.0964,0.1398,0.1398,0.354,100.0,NaN,NaN,NaN,NaN,NaN,exploratorio+similares_2026,0.15
59,"Híbrido: audio+generos+historia, peso=0.15",5,0.0948,0.1387,0.1378,0.351,100.0,NaN,NaN,NaN,NaN,NaN,audio+generos+historia,0.15
65,"Híbrido: todo_historico+tags, peso=0.15",5,0.0944,0.1397,0.1373,0.349,99.0,NaN,NaN,NaN,NaN,NaN,todo_historico+tags,0.15


## 5. Probar con tres semillas y tres tamaños de lista
Rearmamos el catálogo y las matrices con el historial anterior a enero de 2014. Para cada semilla sorteamos hasta 1.000 usuarios y volvemos a entrenar ALS con la configuración elegida. Dentro de cada semilla todos los modelos se evalúan sobre los mismos usuarios. La variación entre semillas refleja tanto la muestra de usuarios como la inicialización de ALS.

Calculamos una lista de diez artistas por usuario y tomamos sus primeros 3, 5 o 10. Así se puede ver qué cambia al alargar la lista sin entrenar un modelo nuevo para cada K.

In [52]:
test = preparar_datos(FECHA_TEST, FIN_TEST)
# Para canciones de artistas que aparecen recién en el nuevo catálogo,
# las seleccionamos ANTES de enero de 2014.
candidatas_test = canciones_historicas(test['artistas'], FECHA_TEST)
grupos_test, detalle_test = construir_features(test, FECHA_TEST, candidatas_test)
pop_test = popularidad(test)


# Estas características dependen del catálogo y se preparan una sola vez.
X_cont = combinar_grupos(grupos_test, CONFIG_CONTENIDO[mejor_contenido])
X_hib = combinar_grupos(grupos_test, CONFIG_CONTENIDO[mejor_hibrido['contenido']])

X_audio = combinar_grupos(grupos_test, CONFIG_CONTENIDO['audio'])
colab_test = preparar_colaborativo(test['matrices']['log'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2014-01-01: matriz (53254, 200), densidad 16.15%, usuarios elegibles 15858
No encontré tracks.parquet; omito las consultas de canciones.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [53]:
resultados_test = []
largo_lista = max(CORTES_K)
for semilla in SEMILLAS:
    test['esperados'] = muestra(test, semilla)
    conf = mejor_als_config
    modelo_als_test = entrenar_als(test['matrices'][conf['senal']],
                                   conf['factores'], conf['alpha'], conf['reg'], semilla)
    vecinos_test = calcular_vecinos(colab_test[3], test['esperados'], mejor_cant_vecinos)
    # Cada función toma un usuario y devuelve su lista usando el modelo ya preparado.
    modelos_test = {
        'Popularidad': lambda u: top_k(pop_test, test['matrices']['log'][u].indices, pop_test, largo_lista),
        'Contenido (audio)': lambda u: recomendar_contenido(test,
            X_audio, u, pop_test, largo_lista),
        f'Contenido ({mejor_contenido})': lambda u: recomendar_contenido(test, X_cont, u, pop_test, largo_lista),
        f'Colaborativo ({mejor_cant_vecinos} vecinos)': lambda u: recomendar_colaborativo(
            test, pop_test, colab_test, vecinos_test, u, mejor_cant_vecinos, largo_lista),
        'ALS seleccionado': lambda u: top_k(puntajes_als(modelo_als_test, u),
            test['matrices']['log'][u].indices, pop_test, largo_lista),
        f'Híbrido ({mejor_hibrido["contenido"]}, {mejor_hibrido["peso"]:.2f})':
            lambda u: recomendar_hibrido(test, modelo_als_test, X_hib, u,
                                         pop_test, mejor_hibrido['peso'], largo_lista),
    }

    for nombre, recomendar in modelos_test.items():
        rankings = {u: recomendar(u) for u in test['esperados']}
        for k in CORTES_K:
            medidas = medir_listas(rankings, test['esperados'], len(test['artistas']), k)
            resultados_test.append(dict(modelo=nombre, semilla=semilla, k=k,
                                        usuarios=len(rankings), **medidas))

tabla_test = pd.DataFrame(resultados_test)
display(tabla_test.round(4))
display(tabla_test.groupby(['modelo', 'k'])[['precision','recall','ndcg','hit_rate','cobertura_pct']]
        .agg(['mean','std']).round(4))

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

,modelo,semilla,k,usuarios,precision,recall,ndcg,hit_rate,cobertura_pct
0,Popularidad,42,3,1000,0.0623,0.0458,0.0731,0.155,17.5
1,Popularidad,42,5,1000,0.0608,0.0729,0.0813,0.232,24.0
2,Popularidad,42,10,1000,0.0530,0.1274,0.0983,0.348,37.0
3,Contenido (audio),42,3,1000,0.0547,0.0454,0.0638,0.146,97.0
4,Contenido (audio),42,5,1000,0.0558,0.0781,0.0762,0.224,100.0
5,Contenido (audio),42,10,1000,0.0536,0.1432,0.0997,0.356,100.0
6,Contenido (todo_historico+tags),42,3,1000,0.0693,0.0578,0.0811,0.183,77.5
7,Contenido (todo_historico+tags),42,5,1000,0.0664,0.0941,0.0920,0.255,87.0
8,Contenido (todo_historico+tags),42,10,1000,0.0603,0.1636,0.1149,0.379,94.0
9,Colaborativo (100 vecinos),42,3,1000,0.0887,0.0676,0.1019,0.221,92.0


precision          recall          \
                                               mean     std    mean     std   
modelo                                 k                                      
ALS seleccionado                       3     0.1000  0.0035  0.0855  0.0043   
                                       5     0.0935  0.0027  0.1342  0.0027   
                                       10    0.0838  0.0025  0.2362  0.0029   
Colaborativo (100 vecinos)             3     0.0830  0.0054  0.0629  0.0040   
                                       5     0.0777  0.0014  0.1038  0.0042   
                                       10    0.0702  0.0035  0.1856  0.0052   
Contenido (audio)                      3     0.0516  0.0040  0.0435  0.0035   
                                       5     0.0519  0.0049  0.0734  0.0065   
                                       10    0.0486  0.0049  0.1307  0.0144   
Contenido (todo_historico+tags)        3     0.0633  0.0090  0.0485  0.0105   
                                       5     0.0613  0.0065  0.0813  0.0124   
                                       10    0.0573  0.0032  0.1477  0.0150   
Híbrido (audio+generos+historia, 0.15) 3     0.1006  0.0084  0.0843  0.0129   
                                       5     0.0932  0.0043  0.1297  0.0079   
                                       10    0.0839  0.0034  0.2324  0.0078   
Popularidad                            3     0.0589  0.0032  0.0449  0.0041   
                                       5     0.0585  0.0036  0.0720  0.0019   
                                       10    0.0524  0.0030  0.1272  0.0018   

                                             ndcg         hit_rate          \
                                             mean     std     mean     std   
modelo                                 k                                     
ALS seleccionado                       3   0.1170  0.0068   0.2487  0.0133   
                                       5   0.1314  0.0049   0.3467  0.0138   
                                       10  0.1654  0.0069   0.5040  0.0120   
Colaborativo (100 vecinos)             3   0.0941  0.0073   0.2043  0.0156   
                                       5   0.1051  0.0045   0.2903  0.0093   
                                       10  0.1318  0.0076   0.4323  0.0166   
Contenido (audio)                      3   0.0600  0.0064   0.1370  0.0101   
                                       5   0.0708  0.0082   0.2103  0.0180   
                                       10  0.0905  0.0109   0.3253  0.0332   
Contenido (todo_historico+tags)        3   0.0719  0.0123   0.1607  0.0268   
                                       5   0.0817  0.0121   0.2347  0.0248   
                                       10  0.1035  0.0123   0.3607  0.0259   
Híbrido (audio+generos+historia, 0.15) 3   0.1183  0.0118   0.2483  0.0261   
                                       5   0.1306  0.0089   0.3390  0.0215   
                                       10  0.1640  0.0091   0.4990  0.0216   
Popularidad                            3   0.0686  0.0039   0.1527  0.0032   
                                       5   0.0781  0.0036   0.2307  0.0032   
                                       10  0.0958  0.0028   0.3457  0.0040   

                                          cobertura_pct          
                                                   mean     std  
modelo                                 k                         
ALS seleccionado                       3        99.6667  0.2887  
                                       5       100.0000  0.0000  
                                       10      100.0000  0.0000  
Colaborativo (100 vecinos)             3        93.3333  1.1547  
                                       5        96.6667  0.5774  
                                       10       99.5000  0.5000  
Contenido (audio)                      3        95.5000  1.5000  
                                       5        99.1667  0.7638  
                 

Para interpretar el resultado, miramos el hit rate con cinco recomendaciones. Un valor de 0,30 significa que 30 de cada 100 usuarios evaluados escucharon después al menos un artista de su lista. La celda siguiente calcula el promedio de las tres semillas para cada modelo.

In [54]:
# Un único promedio por modelo, tomando las tres corridas con K = 5.
resumen_cinco = tabla_test.loc[tabla_test['k'] == 5].groupby('modelo')['hit_rate'].mean()
display((100 * resumen_cinco).sort_values(ascending=False).round(1).to_frame('usuarios_con_acierto_pct'))

,usuarios_con_acierto_pct
modelo,
ALS seleccionado,34.7
"Híbrido (audio+generos+historia, 0.15)",33.9
Colaborativo (100 vecinos),29.0
Contenido (todo_historico+tags),23.5
Popularidad,23.1
Contenido (audio),21.0


## 6. Guardar los resultados
Guardamos tres archivos en `data/parquet/`: las características por artista, el perfil de géneros por usuario y las métricas de cada modelo para cada semilla y tamaño de lista. La tabla de métricas permite volver a comparar resultados sin entrenar otra vez.

In [55]:
# Datos explicables, una fila por artista (catálogo de train de la prueba final).
base = pd.DataFrame({'artist_id': test['artistas']})
base['artist_name'] = base.artist_id.map(nombres_artistas)
base = base.set_index('artist_id')

# Métricas del audio por artista y del historial de entrenamiento.
base = base.join(detalle_test['audio'].add_prefix('audio_'))
base = base.join(test['resumen'].set_index('artist_id').add_prefix('hist_'))
base = base.join(detalle_test['duracion'])

# Géneros agregados: 15 columnas numéricas, no centenares de etiquetas.
generos = pd.DataFrame(detalle_test['generos'], index=test['artistas'],
                       columns=['genero_'+x for x in GENEROS])
generos_tracks = pd.DataFrame(detalle_test['generos_canciones'],
                               index=test['artistas'],
                               columns=['canciones_genero_'+x for x in GENEROS])
base = base.join(generos).join(generos_tracks)
base = base.reset_index().rename(columns={'index': 'artist_id'})
archivo_enriquecido = ruta_parquet / 'artist_features_enriquecidas.parquet'
base.to_parquet(archivo_enriquecido, index=False)

# También guardamos perfiles por usuario para el clustering, siempre calculados
# únicamente con escuchas anteriores a 2014 (no contiene el test).
M = test['matrices']['log']
usuario_n_artistas = test['matrices']['binaria'].getnnz(axis=1)
totales = np.asarray(M.sum(axis=1)).ravel()
perfil_gen = (M @ detalle_test['generos']) / np.maximum(totales[:, None], 1)
perfiles_usuario = pd.DataFrame(perfil_gen, columns=['usuario_genero_'+x for x in GENEROS])
perfiles_usuario.insert(0, 'n_artistas_catalogo', usuario_n_artistas)
perfiles_usuario.insert(0, 'user_id', np.asarray(test['usuarios']))
archivo_usuarios = ruta_parquet / 'user_features_musicales.parquet'
perfiles_usuario.to_parquet(archivo_usuarios, index=False)

archivo_experimentos = ruta_parquet / 'recomendador_resultados_k_semillas.parquet'
tabla_test.to_parquet(archivo_experimentos, index=False)

print('Archivo de características:', archivo_enriquecido)
print('Tabla de comparación:', archivo_experimentos)
print('Artistas guardados:', len(base))
print('Perfiles de usuarios:', archivo_usuarios, '| Usuarios:', len(perfiles_usuario))
base.head(5)

Archivo de características: C:\Users\mgrau\musica-por-pais\data\parquet\artist_features_enriquecidas.parquet
Tabla de comparación: C:\Users\mgrau\musica-por-pais\data\parquet\recomendador_resultados_k_semillas.parquet
Artistas guardados: 200
Perfiles de usuarios: C:\Users\mgrau\musica-por-pais\data\parquet\user_features_musicales.parquet | Usuarios: 53254


,artist_id,artist_name,audio_canciones_con_audio,audio_energy_media,audio_energy_std,audio_valence_media,audio_valence_std,audio_danceability_media,audio_danceability_std,audio_acousticness_media,...,canciones_genero_hip_hop,canciones_genero_electronic,canciones_genero_rnb_soul,canciones_genero_jazz,canciones_genero_blues,canciones_genero_folk_country,canciones_genero_latin,canciones_genero_reggae,canciones_genero_classical,canciones_genero_ambient
0,3,Foo Fighters,103,0.769825,0.215981,0.373432,0.185369,0.433864,0.132614,0.092816,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13,Limp Bizkit,89,0.791697,0.171646,0.429958,0.221472,0.538640,0.156180,0.088153,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,27,Red Hot Chili Peppers,124,0.860532,0.142731,0.505373,0.229110,0.556202,0.140505,0.060438,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,28,Avril Lavigne,134,0.782328,0.184286,0.476987,0.225060,0.515582,0.101215,0.135088,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,29,Rage Against the Machine,75,0.881467,0.119110,0.446531,0.183199,0.444653,0.108806,0.073272,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
